# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [18]:
# Code here
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer

df = pd.read_csv("./Income Dirty Data.csv")
print(df)

       ID     sex  age    income  tax_15_pct
0       1   Women   21  147168.0    22075.20
1       2  Female   29  119595.0    17939.25
2       3  Female   56   87770.0    13165.50
3       4     NaN   21   54259.0     8138.85
4       5    Male   28       NaN   160230.00
..    ...     ...  ...       ...         ...
995   996  Female   36       NaN    14623.80
996   997  Female    0       NaN    19976.25
997   998  Female   59   96847.0    14527.05
998   999     Man   38  108257.0    16238.55
999  1000    Male   49   70424.0    10563.60

[1000 rows x 5 columns]


3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [19]:
# Code here

# Total number of NaN values for each column
print("NaN values in each column:")
print(df.isna().sum())

# Percentage of NaN values in the entire dataset
nan_percentage = df.isna().sum().sum() / df.size * 100
print("\nPercentage of NaN values in the dataset:", nan_percentage, "%")

# Number of rows without any NaN values
complete_rows = df.dropna().shape[0]
print("\nNumber of rows without any NaN values:", complete_rows)

NaN values in each column:
ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64

Percentage of NaN values in the dataset: 5.800000000000001 %

Number of rows without any NaN values: 733


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [20]:
# Check each rule
valid_rows = (
    (df["age"] >= 18) &
    (df["tax_15_pct"] == df["income"] * 0.15) &
    (df["income"] > 0)
)

# Calculate the percentage of rows that follow all rules
percentage_valid = valid_rows.mean() * 100

print("Percentage of data that does not violate any rules:", percentage_valid, "%")

Percentage of data that does not violate any rules: 47.0 %


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [21]:
# Code here

# Code here

df["sex"] = df["sex"].replace({
    "Women": "Female",
    "Woman": "Female",
    "Man": "Male",
    "Men": "Male"
})

print(df["sex"].value_counts(dropna=False))

sex
Male      457
Female    455
NaN        88
Name: count, dtype: int64


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [24]:
# Code here
# 6. Replace non-positive Age values with NaN
df.loc[df["age"] <= 0, "age"] = np.nan

# 7. Replace non-positive Income values with NaN
df.loc[df["income"] <= 0, "income"] = np.nan

# 8. Replace non-positive Tax (15%) values with NaN
df.loc[df["tax_15_pct"] <= 0, "tax_15_pct"] = np.nan

print(df[["age", "income", "tax_15_pct"]])

      age    income  tax_15_pct
0    21.0  147168.0    22075.20
1    29.0  119595.0    17939.25
2    56.0   87770.0    13165.50
3    21.0   54259.0     8138.85
4    28.0       NaN   160230.00
..    ...       ...         ...
995  36.0       NaN    14623.80
996   NaN       NaN    19976.25
997  59.0   96847.0    14527.05
998  38.0  108257.0    16238.55
999  49.0   70424.0    10563.60

[1000 rows x 3 columns]


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [25]:
# Code here
# 9. Use KNNImputer to replace missing values

# Show data before imputing
print("Before imputing:")
print(df.head(10))

# Convert Sex to numeric values
df["sex"] = df["sex"].replace({
    "Female": 0,
    "Male": 1
})

# Create a scaler
scaler = preprocessing.StandardScaler()

# Scale the columns used for KNN
scaled_data = scaler.fit_transform(df[["sex", "age", "income", "tax_15_pct"]])

# Create the KNN imputer
imputer = KNNImputer(n_neighbors=5)

# Impute the missing values
imputed_data = imputer.fit_transform(scaled_data)

# Convert the imputed data back to its original scale
imputed_data = scaler.inverse_transform(imputed_data)

# Put the imputed data back into the DataFrame
df[["sex", "age", "income", "tax_15_pct"]] = imputed_data

# Convert Sex back to Female/Male
df["sex"] = df["sex"].round().replace({
    0: "Female",
    1: "Male"
})

# Show data after imputing
print("\nAfter imputing:")
print(df.head(10))

Before imputing:
   ID     sex   age    income  tax_15_pct
0   1  Female  21.0  147168.0    22075.20
1   2  Female  29.0  119595.0    17939.25
2   3  Female  56.0   87770.0    13165.50
3   4     NaN  21.0   54259.0     8138.85
4   5    Male  28.0       NaN   160230.00
5   6  Female   NaN  128326.0    19248.90
6   7  Female   NaN       NaN         NaN
7   8  Female  24.0       NaN    11820.60
8   9  Female  38.0  149473.0    22420.95
9  10    Male  48.0  113663.0  1136630.00

After imputing:
   ID     sex   age    income  tax_15_pct
0   1  Female  21.0  147168.0    22075.20
1   2  Female  29.0  119595.0    17939.25
2   3  Female  56.0   87770.0    13165.50
3   4    Male  21.0   54259.0     8138.85
4   5    Male  28.0   87077.0   160230.00
5   6  Female  36.2  128326.0    19248.90
6   7  Female  40.8   76111.4    14991.18
7   8  Female  24.0  110944.4    11820.60
8   9  Female  38.0  149473.0    22420.95
9  10    Male  48.0  113663.0  1136630.00


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

It is important to clean a dataset before analyzing it because missing or incorrect information can affect the results. If the data is not cleaned first, the calculations might be inaccurate and could lead to the wrong conclusions. Cleaning the data helps make sure the information is reliable and that the analytics actually represent the data.


### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.